# Laboratorio 05.2 — Búsqueda semántica manual

Implementaremos la ruta desde consulta hasta ranking usando vectores y similitud coseno directamente, antes de introducir abstracciones de frameworks.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Vectorizar documentos y consultas con el mismo espacio de características.
- Calcular cosine similarity y ordenar vecinos manualmente.
- Comparar top-k y relevancia con etiquetas conocidas.
- Explicar por qué búsqueda TF-IDF es un baseline léxico y no entiende toda paráfrasis.

## 2. Conceptos

La búsqueda semántica compara una consulta vectorizada con vectores documentales y devuelve los vecinos más cercanos. La similitud coseno mide orientación, mientras top-k fija cuántos resultados pasan al siguiente componente. Si el vectorizador no conoce términos o no representa sinónimos, el ranking puede ser pobre; la métrica de recuperación debe evaluarse aparte de la generación.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Consulta] --> B[Vectorizador ajustado]
    C[Documentos] --> B
    B --> D[Vectores compatibles]
    D --> E[Cosine similarity]
    E --> F[Orden descendente]
    F --> G[Top-k documentos]
```

## 4. Preparación del entorno

Cargamos los documentos compartidos y las herramientas vectoriales. El ranking se calcula localmente; no se usa LangChain ni un servicio de embeddings remoto.

In [1]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import sys

raices_candidatas = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
raiz_curso = next(
    (
        candidato
        for raiz in raices_candidatas
        for candidato in (raiz / 'genai', raiz)
        if (candidato / 'src' / 'genai').is_dir()
    ),
    None,
)
if raiz_curso is None:
    raise FileNotFoundError('No se encontró el proyecto genAI con src/genai.')

sys.path.insert(0, str(raiz_curso / 'src'))

In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from genai.environment import preparar_entorno
from genai.evaluation.metrics import hit_rate_at_k, mean_reciprocal_rank, precision_at_k, recall_at_k
from genai.retrieval.vector_search import cosine_similarity, rank_by_cosine

raiz_curso = preparar_entorno()
documentos = pd.read_json(raiz_curso / 'data' / 'documents' / 'faq.jsonl', lines=True).to_dict(orient='records')
ids_documentos = [documento['id'] for documento in documentos]

## 5. Definición del problema

Buscaremos evidencia para consultas sobre seguimiento de envíos, cargos duplicados y plazo de devolución. Las etiquetas relevantes se definen manualmente a partir del corpus FAQ sintético compartido.

## 6. Línea base

La línea base puntúa cada documento por cantidad de palabras exactas compartidas. A continuación veremos cómo la normalización TF-IDF y cosine similarity consideran frecuencia y magnitud vectorial.

In [3]:
preguntas = [
    {'id': 'q_envio', 'texto': 'seguimiento paquete retraso entrega', 'relevantes': {'envio_seguimiento', 'envio_entrega'}},
    {'id': 'q_pago', 'texto': 'cargo duplicado movimiento tarjeta', 'relevantes': {'pago_duplicado'}},
    {'id': 'q_devolucion', 'texto': 'devolucion 30 dias entrega producto', 'relevantes': {'devolucion_plazo'}},
]
def contar_terminos_compartidos(consulta: str, texto: str) -> int:
    """Cuenta terminos literales compartidos tras normalizar mayusculas."""
    return len(set(consulta.casefold().split()) & set(texto.casefold().split()))

consulta_base = preguntas[0]['texto']
base_lexica = pd.DataFrame([
    {'id': documento['id'], 'coincidencias': contar_terminos_compartidos(consulta_base, documento['texto'])}
    for documento in documentos
])
display(base_lexica.sort_values('coincidencias', ascending=False).head(4))

,id,coincidencias
0,envio_seguimiento,2
1,envio_entrega,1
2,devolucion_plazo,0
3,devolucion_reembolso,0


## 7. Implementación

Ajustamos un único vectorizador sobre documentos y transformamos las consultas con el mismo vocabulario. Calculamos la fórmula coseno manualmente, ordenamos puntuaciones y después contrastamos el resultado con la utilidad de ranking del paquete.

In [4]:
vectorizador = TfidfVectorizer(stop_words=['el', 'la', 'los', 'las', 'de', 'del', 'un', 'una', 'y', 'si'])
matriz_documentos = vectorizador.fit_transform([documento['texto'] for documento in documentos]).toarray()

def buscar_manualmente(consulta: str, top_k: int) -> list[tuple[str, float]]:
    """Vectoriza, calcula coseno y retorna IDs ordenados por similitud."""
    vector_consulta = vectorizador.transform([consulta]).toarray()[0]
    productos = matriz_documentos @ vector_consulta
    normas = np.linalg.norm(matriz_documentos, axis=1) * np.linalg.norm(vector_consulta)
    similitudes = np.divide(productos, normas, out=np.zeros_like(productos), where=normas != 0)
    indices = np.argsort(-similitudes, kind='stable')[:top_k]
    return [(ids_documentos[indice], float(similitudes[indice])) for indice in indices]

ranking_manual = buscar_manualmente(preguntas[0]['texto'], top_k=3)
vector_consulta = vectorizador.transform([preguntas[0]['texto']]).toarray()[0]
ranking_utilidad = rank_by_cosine(vector_consulta, matriz_documentos, top_k=3)
assert [ids_documentos[indice] for indice, _ in ranking_utilidad] == [doc_id for doc_id, _ in ranking_manual]
display(pd.DataFrame(ranking_manual, columns=['documento_id', 'similitud_coseno']))

,documento_id,similitud_coseno
0,envio_seguimiento,0.312540
1,envio_entrega,0.220913
2,devolucion_plazo,0.116038


## 8. Experimentos

Repetimos las mismas consultas con top-k de 1, 3 y 5; no cambia el vectorizador ni las consultas. Hit Rate, Precision y Recall responden preguntas distintas sobre la evidencia recuperada.

In [5]:
filas_retrieval = []
for pregunta in preguntas:
    for k in (1, 3, 5):
        ranking = buscar_manualmente(pregunta['texto'], top_k=k)
        ids = [doc_id for doc_id, _ in ranking]
        filas_retrieval.append({
            'consulta_id': pregunta['id'],
            'k': k,
            'hit_rate': hit_rate_at_k(ids, pregunta['relevantes'], k),
            'precision_at_k': precision_at_k(ids, pregunta['relevantes'], k),
            'recall_at_k': recall_at_k(ids, pregunta['relevantes'], k),
            'reciprocal_rank': 1 / (next((rango for rango, doc_id in enumerate(ids, start=1) if doc_id in pregunta['relevantes']), float('inf'))),
            'primer_resultado': ids[0],
        })
resultados_retrieval = pd.DataFrame(filas_retrieval)
display(resultados_retrieval)
resultados_retrieval.to_csv(raiz_curso / 'outputs' / '05_02_manual_search.csv', index=False)

,consulta_id,k,hit_rate,precision_at_k,recall_at_k,reciprocal_rank,primer_resultado
0,q_envio,1,1.0,1.000000,0.5,1.0,envio_seguimiento
1,q_envio,3,1.0,0.666667,1.0,1.0,envio_seguimiento
2,q_envio,5,1.0,0.400000,1.0,1.0,envio_seguimiento
3,q_pago,1,1.0,1.000000,1.0,1.0,pago_duplicado
4,q_pago,3,1.0,0.333333,1.0,1.0,pago_duplicado
5,q_pago,5,1.0,0.200000,1.0,1.0,pago_duplicado
6,q_devolucion,1,1.0,1.000000,1.0,1.0,devolucion_plazo
7,q_devolucion,3,1.0,0.333333,1.0,1.0,devolucion_plazo
8,q_devolucion,5,1.0,0.200000,1.0,1.0,devolucion_plazo


## 9. Evaluación

Calculamos MRR sobre el ranking completo y resumimos métricas por k. La función `recall_at_k` requiere al menos un documento relevante por consulta; por eso cada consulta del dataset tiene una etiqueta no vacía.

In [6]:
ranking_completo = [
    [doc_id for doc_id, _ in buscar_manualmente(pregunta['texto'], top_k=len(documentos))]
    for pregunta in preguntas
]
relevancia = [pregunta['relevantes'] for pregunta in preguntas]
mrr = mean_reciprocal_rank(ranking_completo, relevancia)
resumen_metricas = resultados_retrieval.groupby('k', as_index=False).agg(
    hit_rate=('hit_rate', 'mean'),
    precision=('precision_at_k', 'mean'),
    recall=('recall_at_k', 'mean'),
)
print(f'MRR: {mrr:.3f}')
display(resumen_metricas)
resumen_metricas.to_csv(raiz_curso / 'outputs' / '05_02_retrieval_metrics.csv', index=False)

MRR: 1.000


,k,hit_rate,precision,recall
0,1,1.0,1.000000,0.833333
1,3,1.0,0.444444,1.000000
2,5,1.0,0.266667,1.000000


## 10. Discusión

Top-k mayor suele aumentar cobertura y reducir precisión porque incorpora más documentos. Un ranking con buena similitud puede ser irrelevante si las etiquetas están mal definidas. El vectorizador léxico no entiende intención; embeddings entrenados pueden ayudar, pero también deben evaluarse sobre consultas reales.

## 11. Ejercicios

1. Añade una consulta sin términos compartidos y analiza su ranking.
2. Cambia la lista de stop words y observa qué documentos suben.
3. Compara cosine con producto punto y explica el efecto de normalizar vectores.
4. Diseña una consulta con dos documentos relevantes y calcula a mano Precision@3.

## 12. Desafío

Sustituye TF-IDF por embeddings densos y conserva exactamente corpus, consultas y etiquetas. Compara Hit Rate, Precision, Recall y MRR; analiza fallos antes de cambiar el generador de respuestas.

## 13. Ideas clave

- La búsqueda manual son vectorización, puntuación y ordenamiento.
- Top-k es una decisión de contexto y debe evaluarse.
- Hit Rate, Precision, Recall y MRR miden aspectos distintos.
- Evalúa recuperación independientemente de la generación.